# Assignment: Download, Run, and Fine-Tune a Real Model

**Unit:** Training & Tradeoffs | **Companion reading:** Lessons 1–5  
**Suggested time:** 2–3 hours (Part 1–2 in session one; Part 3–4 in session two)

---

## What is this assignment trying to do?

You have used ChatGPT-style products. This assignment shows **what happens underneath** when a model becomes useful for a specific job.

In four steps you will:

1. **Download** a real pretrained model from the internet (you are not building one from scratch).
2. **Run** it and observe a raw **base model**, which often *continues* text instead of *following instructions*.
3. **Fine-tune** it on examples of one narrow task (starter task: translate plain English into Yoda-speak).
4. **Track** training with **Weights & Biases (W&B)** so you can compare experiments the way ML teams do.

**Big idea:** Pretraining teaches general language (Lesson 1: How Do LLMs Actually Learn). Fine-tuning nudges the same weights toward *your* task using a small dataset and a short training run. You are doing a miniature version of what product teams do before launch.

**Model:** `distilgpt2` (~82 million parameters). Small enough to fine-tune in minutes on Colab.

| Part | What you do | Why |
|------|-------------|-----|
| 1 | Load base model, generate text | See "before" behavior |
| 2 | Load the Yoda dataset (720 sentence pairs) | Give the model teaching material |
| 3 | Write the training function, train twice, log to W&B | Practice hyperparameters + experiment tracking |
| 4 | Compare before/after | Show whether fine-tuning helped |

---
## Key terms (read this first)

| Term | Plain-English meaning |
|------|------------------------|
| **Pretrained model** | Already trained on huge text. You download weights instead of training from zero. |
| **Base model** | Pretrained but not customized for your task. Strong at text completion. |
| **Fine-tuning** | Extra training on *your* examples to specialize behavior. |
| **RLHF (Reinforcement Learning from Human Feedback)** | A further training stage, after fine-tuning, that uses human preferences to shape responses. Part of how ChatGPT-style assistants are built; `distilgpt2` has none of this. |
| **Hugging Face** | Hub + Python library for models, tokenizers, and datasets. |
| **Tokenizer** | Converts text to numbers (tokens) the model can process. |
| **Prompt** | Text you provide before the model generates more text. |
| **Generate** | Model predicts upcoming tokens and returns new text. |
| **Loss** | Error score during training. **Lower is better.** |
| **Epoch** | One full pass through all training examples. |
| **Learning rate** | Size of each weight update step. |
| **Batch size** | Number of examples processed before one update. |
| **Weights & Biases (W&B)** | Free website that logs loss curves and settings for each training run. |
| **Checkpoint** | Saved model file after training (your fine-tuned model). |

**Analogy:** Pretraining = reading the whole library. Fine-tuning = practicing one interview format. W&B = lab notebook that records every practice session.

---
## Part 0: Setup

### Where to run

**Google Colab is recommended** (free notebook + optional GPU).

1. Upload this file to Colab.
2. **Runtime → Change runtime type → T4 GPU** before Part 3.
3. Run cells top to bottom.

### Libraries

| Library | Purpose |
|---------|---------|
| `transformers` | Load Hugging Face models; run training |
| `datasets` | Hold your training examples |
| `accelerate` | Required helper for `Trainer` |
| `wandb` | Log metrics to Weights & Biases |
| `torch` | PyTorch (deep learning engine) |

Run the next cell. In Colab, uncomment `pip install` if imports fail.

In [ ]:
# Colab: uncomment if imports fail
# !pip install -q transformers datasets accelerate wandb

import os
import random
import textwrap
import time

import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("Tip: enable Colab GPU before Part 3 for faster training.")

MODEL_NAME = "distilgpt2"


---
## Part 1: Download and Run a Pretrained Model

### What you are doing (no training yet)

1. **Download** weights from Hugging Face (~350 MB for distilgpt2).
2. **Load** a matching **tokenizer** (text ↔ numbers).
3. **Generate** text: the model predicts likely next tokens.

`distilgpt2` learned **next-token prediction** on web text. It is **not** instruction-tuned like ChatGPT. When you write "Write a haiku," it may repeat the question; that is expected.

**Save these outputs**, because Part 4 compares base vs fine-tuned on similar prompts.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
base_model.to(device)
base_model.eval()

print(f"Loaded {MODEL_NAME}")
print(f"Parameters: {base_model.num_parameters() / 1e6:.1f} million")


In [ ]:
def generate_text(model, prompt, max_new_tokens=80, temperature=0.8):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


BASELINE_PROMPTS = [
    "Once upon a time",
    "Question: What is the capital of France?\nAnswer:",
    "Write a haiku about rain.",
    "Translate to Yoda-speak: The Force is strong in you.\n",
]

print("=== Base model outputs - SAVE for Part 4 ===\n")
for p in BASELINE_PROMPTS:
    result = generate_text(base_model, p)
    print("PROMPT:", repr(p))
    print(textwrap.fill(result, width=100))
    print("-" * 60)


### Part 1 reflection

**Question:** Does the model follow instructions, or mostly continue the prompt? Why?

Answer in **2-4 sentences** in the next cell.

In [ ]:
part1_reflection = ""  # TODO


---
## Part 2: The Fine-Tuning Dataset

### What is fine-tuning data?

Each training row has:
- **Input** is what a user might type
- **Output** is the answer you want

We use the [`dvgodoy/yoda_sentences`](https://huggingface.co/datasets/dvgodoy/yoda_sentences) dataset from the Hugging Face Hub (used in [this fine-tuning tutorial](https://huggingface.co/blog/dvgodoy/fine-tuning-llm-hugging-face)). It has **720** plain English sentences and their Yoda-speak translations:

| Column | Example |
|--------|---------|
| `sentence` (input) | The birch canoe slid on the smooth planks. |
| `translation_extra` (output) | On the smooth planks, the birch canoe slid. Yes, hrrrm. |

We format each pair as one string:

```
### Input:
The birch canoe slid on the smooth planks.

### Output:
On the smooth planks, the birch canoe slid. Yes, hrrrm.
```

The model reads the full string and learns patterns. After enough examples, it learns: after `### Output:`, rewrite the input in Yoda-speak.

We hold out 10% of the sentences (never trained on) so you can test the fine-tuned model on text it has not seen. Nothing to complete here; run the cells and read the examples.

In [ ]:
TASK_NAME = "yoda"
DATASET_NAME = "dvgodoy/yoda_sentences"
INPUT_COLUMN = "sentence"
OUTPUT_COLUMN = "translation_extra"

raw_dataset = load_dataset(DATASET_NAME, split="train")
splits = raw_dataset.train_test_split(test_size=0.1, seed=SEED)
train_data, eval_data = splits["train"], splits["test"]

print(f"Train: {len(train_data)} examples | Held out: {len(eval_data)} examples")
for ex in train_data.select(range(3)):
    print(ex[INPUT_COLUMN], "->", ex[OUTPUT_COLUMN])

In [ ]:
PROMPT_TEMPLATE = "### Input:\n{input}\n\n### Output:\n{output}"


def format_training_text(input_text, output_text):
    return PROMPT_TEMPLATE.format(input=input_text, output=output_text)


def format_inference_prompt(input_text):
    return f"### Input:\n{input_text}\n\n### Output:\n"


def build_dataset(examples):
    texts = [format_training_text(ex[INPUT_COLUMN], ex[OUTPUT_COLUMN]) for ex in examples]
    return Dataset.from_dict({"text": texts})


print(build_dataset(train_data)[0]["text"])

---
## Part 3: Fine-Tune the Model

### What fine-tuning does

1. Start from pretrained distilgpt2.
2. Show your formatted examples for several **epochs**.
3. Update weights with **gradient descent** to lower **loss** (same core idea as Lesson 1: How Do LLMs Actually Learn).
4. Save a **checkpoint** you reload like the original model.

This takes **minutes** on Colab, not the months/dollars of pretraining.

### Hyperparameters (you choose these)

| Setting | Meaning | Start with |
|---------|---------|------------|
| Learning rate | Update step size | `5e-5` and `2e-4` in two runs |
| Epochs | Passes through data | 3 |
| Batch size | Examples per update | 8 (use 4 if OOM) |

Run **two experiments** with different learning rates. Training uses all 648 training sentences, so each run takes a few minutes on a T4.

---

## What is Weights & Biases (W&B)?

**Weights & Biases** is an experiment-tracking website used in real ML work.

**Without W&B:** You change learning rate, retrain, forget what you tried, repeat mistakes.

**With W&B:**
- Plots **training loss** over time (line chart)
- Stores **hyperparameters** next to each run
- Lets you **compare runs** side by side

**Setup steps:**
1. Create account: https://wandb.ai
2. Get API key: https://wandb.ai/authorize
3. You will log in from the notebook in TODO 3 below (run the login cell and paste your key when prompted).
4. After training, open your project on wandb.ai and look at your two loss curves.
5. **Save one plot:** choose any chart from your W&B page that you think shows your training best (for example the training loss for both runs). Save it as an image: use the chart's download/export option, or just take a screenshot. You will submit this image together with your `.py` file.

If login fails, ask on Piazza about W&B issues.

---

### TODO 1: set hyperparameters. TODO 2: write the training function. TODO 3: run training.

In [1]:
LEARNING_RATE_RUN1 = None 
LEARNING_RATE_RUN2 = None 
NUM_EPOCHS = None         
BATCH_SIZE = None         
MAX_LENGTH = 128

### TODO 2: Write the training function

The data pipeline is done for you: `train_finetune_run` tokenizes the examples, builds a data collator and loads a fresh copy of the pretrained model. You write the part that actually trains it. Inside the function:

1. Create `TrainingArguments(...)` with:

| Argument | Value |
|----------|-------|
| `output_dir` | the `output_dir` parameter |
| `num_train_epochs` | `NUM_EPOCHS` |
| `per_device_train_batch_size` | `BATCH_SIZE` |
| `learning_rate` | the `learning_rate` parameter |
| `report_to` | `"wandb"` |
| `run_name` | the `run_name` parameter |
| `logging_steps` | `10` |
| `save_strategy` | `"no"` |
| `seed` | `SEED` |
| `fp16` | `torch.cuda.is_available()` |

2. Create a `Trainer` with the `model`, the `args`, the tokenized `dataset` (as `train_dataset`) and the `collator` (as `data_collator`).
3. Call `trainer.train()`.
4. Save the model with `trainer.save_model(output_dir)` and the tokenizer with `tokenizer.save_pretrained(output_dir)`.

Docs: [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer).

In [ ]:
def tokenize_batch(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH)


def train_finetune_run(train_data, learning_rate, output_dir, run_name):
    if learning_rate is None or NUM_EPOCHS is None or BATCH_SIZE is None:
        raise ValueError("Complete TODO 1 first.")

    dataset = build_dataset(train_data).map(tokenize_batch, batched=True, remove_columns=["text"])
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

    # TODO 2a: create TrainingArguments
    args = None

    # TODO 2b: create the Trainer
    trainer = None

    start = time.time()
    # TODO 2c: train
    elapsed = time.time() - start

    # TODO 2d: save the model and the tokenizer to output_dir
    print(f"Saved {output_dir} in {elapsed/60:.1f} min - check W&B for loss curve")
    return model, elapsed

### TODO 3: Run two training runs

1. Finish TODO 1 and TODO 2.
2. Run the login cell below and paste your API key.
3. Uncomment and run the training cell after it.
4. Open W&B; you should see **two runs** and two loss curves.
5. Pick the smoother/lower-loss run for Part 4.

**Reading loss:** Should generally decrease. Flat or spiking lines often mean a bad learning rate.

In [ ]:
import wandb
wandb.login()

In [ ]:
finetuned_model_1, time_1 = train_finetune_run(
    train_data, LEARNING_RATE_RUN1, "./finetuned-run1", f"{TASK_NAME}-lr{LEARNING_RATE_RUN1}")
finetuned_model_2, time_2 = train_finetune_run(
    train_data, LEARNING_RATE_RUN2, "./finetuned-run2", f"{TASK_NAME}-lr{LEARNING_RATE_RUN2}")

---
## Part 4: Evaluate and Reflect

Compare **base** vs **fine-tuned** on prompts **not** in training data.

For fine-tuned runs, use `format_inference_prompt()` so the format matches training.

**Deliverables:**
- Two logged W&B runs (describe your loss curves in the reflection)
- One plot image from your W&B page, your choice (submitted with your `.py`; see Part 5)
- 3–5 side-by-side comparisons
- A short reflection (about 150 words, well under a page; questions in the last cell)

In [ ]:
FINETUNED_DIR = "./finetuned-run1"  # switch if run 2 was better

try:
    finetuned_model = AutoModelForCausalLM.from_pretrained(FINETUNED_DIR).to(device)
    finetuned_model.eval()
    print("Loaded", FINETUNED_DIR)
except Exception as e:
    print("Train in Part 3 first.", e)
    finetuned_model = None


In [ ]:
# Held-out sentences the model never trained on, plus two it should not know how to handle
EVAL_EXAMPLES = [(ex[INPUT_COLUMN], ex[OUTPUT_COLUMN]) for ex in eval_data.select(range(3))] + [
    ("The Force is strong in you!", None),
    ("Write a poem about the ocean.", None),
]


def first_response(text):
    # keep only what follows "### Output:" up to the first line break
    return text.split("### Output:\n")[-1].split("\n")[0].strip()


for prompt, expected in EVAL_EXAMPLES:
    print("INPUT:", prompt)
    if expected:
        print("EXPECTED:", expected)
    print("\nBASE:")
    print(textwrap.fill(generate_text(base_model, prompt, max_new_tokens=40), width=100))
    if finetuned_model:
        print("\nFINE-TUNED:")
        print(first_response(generate_text(finetuned_model, format_inference_prompt(prompt), max_new_tokens=40)))
    print("=" * 72)

### Part 4 reflection

Answer both questions in the reflection below:

1. Did fine-tuning help? Point to a specific comparison above.
2. What did your W&B loss curves look like for the two learning rates?

In [ ]:
# Did fine-tuning help? What did the loss curves look like?

part4_reflection = ""
print("Fill in the reflection before submitting.")

---
## Part 5: Submit as a `.py` File

So far, you have been working in a `.ipynb` notebook because it is convenient for development; you can run one cell at a time, see outputs immediately, and experiment freely. For grading, we need a plain `.py` script instead, since it is easier for us to run and review outside a Jupyter environment.

### Before converting

1. Make sure every TODO is complete: no `None` hyperparameters, a working `train_finetune_run`, a saved W&B plot image, no empty `part1_reflection`, and a filled-in `part4_reflection` that actually describes your loss curves and results.
2. Run the whole notebook top to bottom and confirm there are no errors.

### Convert to `.py`

- **Google Colab:** **File → Download → Download .py**
- **VS Code:** click the **"..."** menu at the top of the notebook → **Export** → **Python Script**
- **Using something else?** Search how to export your tool's notebook to `.py`, or ask on Piazza.

### Submit

Rename the file to `firstname_lastname_fine_tune.py` and submit it **together with your W&B plot image** (`.png` or `.jpg`), two files in total. Both reflections are already saved inside the `.py` as `part1_reflection` and `part4_reflection`; there is nothing else to attach.

### About grading

The autograder gives you an instant score when you submit, but **that score is not final**. We (the instructors) will manually review every submission, including your code, your reflections and your W&B plot, before grades are finalized, so your final score may differ from what the autograder shows.